In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import torch

from helmholtz_eos_net.test import load_model_from_folder


msize=12
mwidth = 2
fsize=16
alpha=0.3
lsize = 2
framewidth = 3
markeredgewidth=3
markeredgewidth_no=10

print('Imports successful!')

In [ ]:

model_folder = "ray_results/d29fd_00000/"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, config = load_model_from_folder(model_folder, device)
model.eval()

In [ ]:
substance_paras = pd.read_csv("extrapolated_parameters.csv")

substance_paras

In [ ]:
from feos.pcsaft import PcSaftParameters, PcSaftRecord, Identifier, PureRecord
from feos.eos import EquationOfState, State, PhaseDiagram
from feos.eos import Contributions
from si_units import * 


eos_keys = ["m","sigma","epsilon_k","kappa_ab","epsilon_k_ab","mu"]
embedding_keys = ["molarweight","m","sigma","epsilon_k","kappa_ab","epsilon_k_ab","mu"]

vle_df_dict = {}

smiles_list = substance_paras["name"].tolist()

for index, row in substance_paras.iterrows():
    smiles = row["name"]
    sub = row

    M = 1*(GRAM/MOL)
    m = sub["m"]

    identifier = Identifier( cas="000-00-0", name="dummy" )
    saftrec = PcSaftRecord( m=sub["m"], sigma=sub["sigma"], epsilon_k=sub["epsilon_k"],
                          kappa_ab=sub["kappa_ab"],epsilon_k_ab=sub["epsilon_k_ab"],mu=sub["mu"],
                           na=0, nb=0 )

    pr = PureRecord(identifier=identifier,model_record=saftrec,molarweight=1)

    saft_paras = PcSaftParameters.from_records( [pr], np.array([[0.0]] ) )

    eos = EquationOfState.pcsaft(saft_paras)

    saft_vle = PhaseDiagram.pure(eos, min_temperature=300*KELVIN , npoints=100)

    temperature_saft = saft_vle.vapor.temperature / KELVIN
    pressure_saft = saft_vle.vapor.pressure / PASCAL
    rho_vap_saft = saft_vle.vapor.density / MOL * METER**3
    rho_liq_saft = saft_vle.liquid.density / MOL * METER**3
    h_vap = saft_vle.vapor.molar_enthalpy(Contributions.Residual)
    h_liq = saft_vle.liquid.molar_enthalpy(Contributions.Residual)
    dh_vap_saft = (h_vap - h_liq) /JOULE *MOL

    vle_df = pd.DataFrame({
        "temperature_K": temperature_saft,
        "pressure_sat_Pa": pressure_saft,
        "molar_density_vapor_mol_m3": rho_vap_saft,
        "molar_density_liquid_mol_m3": rho_liq_saft,
        "molar_enthalpy_vapor_J_mol": h_vap,
        "molar_enthalpy_liquid_J_mol": h_liq,
        "molar_enthalpy_diff_J_mol": dh_vap_saft
    })

    for mp in config["parameter_columns"]:
        #try:
        #    x = substance_paras.iloc[0][mp]
        #except:
        #    x = substance_paras[mp]
        vle_df[mp] = row[mp]

    vle_df_dict[smiles] = vle_df


In [ ]:
row

In [ ]:
vle_df

In [ ]:
T_step = 5.0

temps_dict = {}

for smiles in smiles_list:
    t_saft = vle_df_dict[smiles]["temperature_K"]
    T_max  = t_saft.max() * 0.9
    T_min  = t_saft.max() * 0.65
    temps_dict[smiles] = np.arange(T_min, T_max, T_step)
    print(f'{smiles}: {t_saft.min():.1f}-{t_saft.max():.1f} K, {len(temps_dict[smiles])} points ({T_min:.1f}-{T_max:.1f} K, step={T_step} K)')


In [ ]:
rho_gas_init_dict = {}
rho_liq_init_dict = {}
T_tensor_dict = {}
branch_input_dict = {}

for smiles in smiles_list:
    vle_df = vle_df_dict[smiles]
    temps = temps_dict[smiles]

    vle_sorted = vle_df.sort_values('temperature_K')
    f_gas = interp1d(vle_sorted['temperature_K'], vle_sorted['molar_density_vapor_mol_m3'],  kind='linear', fill_value='extrapolate')
    f_liq = interp1d(vle_sorted['temperature_K'], vle_sorted['molar_density_liquid_mol_m3'], kind='linear', fill_value='extrapolate')

    rho_gas_init_dict[smiles] = torch.tensor( f_gas(temps)[:, None], dtype=torch.float32)
    rho_liq_init_dict[smiles] = torch.tensor( f_liq(temps)[:, None], dtype=torch.float32)
    T_tensor_dict[smiles]     = torch.tensor(temps[:, None], dtype=torch.float32)

    branch_input_dict[smiles] = torch.tensor(
        np.tile(vle_df[model.substance_params.parameter_columns].iloc[0].values, (len(temps), 1)),
        dtype=torch.float32
    )

    print(f'{smiles}: T_tensor {T_tensor_dict[smiles].shape}, branch_input {branch_input_dict[smiles].shape}')


In [ ]:
model.substance_params.parameter_columns

In [ ]:
branch_input_dict


In [ ]:
if False:
    branch_input[:,0] = branch_input[:,0] + 0.01
    branch_input[:,1] = branch_input[:,1] + 0.01
    branch_input[:,2] = branch_input[:,2] + 0.01

    branch_input[:,0] = branch_input[:,0] - 0.03
    branch_input[:,1] = branch_input[:,1] - 0.02
    branch_input[:,2] = branch_input[:,2] - 0.01

# Run batch VLE flash
with torch.enable_grad():
    p_sat_pred, mu_sat_pred, rho_gas_vle, rho_liq_vle, converged = model.find_vle_flash_batch(
        branch_input, T_tensor, rho_gas_init, rho_liq_init,
        n_steps=40, tol=1e-6, alpha=0.5,
        # determine_rho_liq=0*torch.ones(T_tensor.shape[0], 1)  # freeze liquid at rho_liq_init
    )

rho_gas_vle = rho_gas_vle.detach().numpy().squeeze()
rho_liq_vle = rho_liq_vle.detach().numpy().squeeze()
converged   = converged.numpy()
p_sat_pred  = p_sat_pred.detach().numpy().squeeze()

temperature = T_out.numpy()

print(f'Converged: {converged.sum()}/{len(converged)}')

In [ ]:
results_dict = {}

for smiles in smiles_list:
    branch_input = branch_input_dict[smiles]
    T_tensor = T_tensor_dict[smiles]

    p_sub = branch_input[0:1]       # (1, 8) — single substance for all steps
    T_1d  = T_tensor.squeeze()      # (n_T,) — any order, sorted internally

    rho_l0 = float(rho_liq_init_dict[smiles][-1])
    rho_g0 = float(rho_gas_init_dict[smiles][-1])

    with torch.enable_grad():
        T_out, p_sat_out, p_sat_cc_out, rho_liq_out, rho_gas_out, dHvap_out, converged_out = \
            model.find_vle_curve(p_sub, T_1d, rho_l0, rho_g0, n_steps=40)

    results_dict[smiles] = dict(
        temperature=T_out.numpy(),
        rho_gas_vle=rho_gas_out.detach().numpy(),
        rho_liq_vle=rho_liq_out.detach().numpy(),
        p_sat_pred=p_sat_out.detach().numpy(),
        p_sat_cc=p_sat_cc_out.detach().numpy(),
        delta_H_vap=dHvap_out.detach().numpy(),
        converged=converged_out.detach().numpy(),
    )

    print(f"{smiles}: converged {results_dict[smiles]['converged'].sum()}/{len(results_dict[smiles]['converged'])}")


In [ ]:
colors = plt.cm.tab10(np.linspace(0, 1, len(smiles_list)))
colors = ["orange","cornflowerblue","magenta", "green"]

colors = ["orange", "gold", "purple", "magenta", "forestgreen",  "limegreen", "cornflowerblue" ]


for smiles, color in zip(smiles_list, colors):
    vle_df = vle_df_dict[smiles]
    res = results_dict[smiles]
    temperature = res["temperature"]
    p_sat_pred = res["p_sat_pred"]
    name = smiles

    f_p_vap = interp1d( vle_df.temperature_K, np.log(vle_df.pressure_sat_Pa), fill_value="extrapolate" )
    ref = np.exp(f_p_vap(temperature))
    error = np.mean( np.abs( p_sat_pred -  ref ) / ref )

    if smiles == smiles_list[0]: 
        plt.plot( vle_df.temperature_K, np.log(vle_df.pressure_sat_Pa), "--k", label=f"PCP-SAFT")

    plt.plot( temperature, np.log(p_sat_pred), ".", color=color, markersize=msize,
              label=f"{error*100:.2f}%")
     
    plt.plot( vle_df.temperature_K, np.log(vle_df.pressure_sat_Pa), "--", color=color)      

plt.xlabel(r"temperature / K",fontsize=fsize)
plt.ylabel(r"$ln(p ~/~ Pa)$",fontsize=fsize)

plt.legend(fontsize=fsize, frameon=False)

plt.xticks(fontsize=fsize)  
plt.yticks(fontsize=fsize)  

plt.savefig(model_folder+"VLE_extra_pressure.png",bbox_inches='tight')
plt.savefig(model_folder+"VLE_extra_pressure.pdf",bbox_inches='tight')


In [ ]:

for smiles, color in zip(smiles_list, colors):
    vle_df = vle_df_dict[smiles]
    res = results_dict[smiles]
    temperature = res["temperature"]
    rho_gas_vle = res["rho_gas_vle"]
    rho_liq_vle = res["rho_liq_vle"]
    name = smiles


    f_rho_vap = interp1d( vle_df.temperature_K, vle_df.molar_density_vapor_mol_m3)
    f_rho_liq = interp1d( vle_df.temperature_K, vle_df.molar_density_liquid_mol_m3)

    ref = f_rho_vap(temperature)
    error_vap = np.mean( np.abs( rho_gas_vle -  ref) / ref )

    ref = f_rho_liq(temperature)
    error_liq = np.mean( np.abs( rho_liq_vle -  ref) / ref )

    if smiles == smiles_list[0]: 
        plt.plot(vle_df.molar_density_vapor_mol_m3/1000, vle_df.temperature_K, "--k",label=f"PCP-SAFT")

    plt.plot(rho_gas_vle/1000, temperature, ".", color=color, markersize=msize )
    plt.plot(rho_liq_vle/1000, temperature, ".", color=color, markersize=msize,
              label=f"{error_liq*100:.2f}%")

    plt.plot(vle_df.molar_density_vapor_mol_m3/1000, vle_df.temperature_K, "--", color=color)
    plt.plot(vle_df.molar_density_liquid_mol_m3/1000, vle_df.temperature_K, "--", color=color)


plt.xlabel(r"$\rho~/~kmol/m^3$",fontsize=fsize)
plt.ylabel(r"temperature / K",fontsize=fsize)

plt.legend(fontsize=fsize, frameon=False, loc="lower right")

plt.xticks(fontsize=fsize)  
plt.yticks(fontsize=fsize)  

plt.savefig(model_folder+"VLE_extra_density.png",bbox_inches='tight')
plt.savefig(model_folder+"VLE_extra_density.pdf",bbox_inches='tight')


In [ ]:
path = "full_data_7/saft_parameters.csv"

data = pd.read_csv(path)
data.columns

In [ ]:
data.shape, sum(data["is_associating"] == 1 ), sum(data["has_dipole"] == 1 )

In [ ]:
cols = [ 'm_norm', 'sigma_norm', 'epsilon_k_norm', 'is_associating',
       'has_dipole', 'mu_val', 'kappa_ab_val', 'epsilon_k_ab_val' ]

#cols = [ 'm_norm', 'sigma_norm', 'epsilon_k_norm', 
#        'mu_val', 'kappa_ab_val', 'epsilon_k_ab_val' ]

df = data[cols]

from sklearn.preprocessing import StandardScaler

x = df.to_numpy()
sc = StandardScaler()
x = sc.fit_transform(x)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)

principalComponents = pca.fit_transform(x)

In [ ]:
plt.scatter(principalComponents[:, 0], principalComponents[:, 1], alpha=0.2, marker=".", color="gray")

xx = sc.transform(substance_paras[cols].to_numpy())
xx = pca.transform(xx)

for i,sxx in enumerate(xx):
    plt.scatter(sxx[0], sxx[1], c=colors[i], marker='x', s=100, label=smiles_list[i])
plt.legend()


In [ ]:
cols = [ 'm', 'sigma', 'epsilon_k', 
        'mu', 'kappa_ab', 'epsilon_k_ab' ]

df = data[cols]
x = df.to_numpy()

pca = PCA(n_components=2)

principalComponents = pca.fit_transform(x)
x = StandardScaler().fit_transform(x)

plt.scatter(principalComponents[:, 0], principalComponents[:, 1])

xx = pca.transform(substance_paras[cols].to_numpy())

for i,sxx in enumerate(xx):
    plt.scatter(sxx[0], sxx[1], c=colors[i], marker='x', s=100, label=smiles_list[i])
plt.legend()